In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
import numpy as np
from tqdm import tqdm
import pandas as pd
import os

In [2]:
data_root = 'confirmed_fronts'
data_list = []

for root, dirs, files in os.walk(data_root):
    for file in files:
        if file.lower().endswith('.jpg'):
            parts = file.split('$$')
            if len(parts) >= 4:
                color = parts[3]
                if color and color.lower() != 'unknown':
                    data_list.append({
                        'img_path': os.path.join(root, file),
                        'color': color
                    })

df_final = pd.DataFrame(data_list)

if not df_final.empty:
    le = LabelEncoder()
    df_final['label'] = le.fit_transform(df_final['color'])
    print(f"Всего найдено корректных изображений: {len(df_final)}")
    print(f"Количество уникальных цветов: {len(le.classes_)}")
    class_names = le.classes_
else:
    print("Не удалось извлечь цвета.")

Всего найдено корректных изображений: 61827
Количество уникальных цветов: 23


In [3]:
df_final = df_final[df_final['color'] != 'Unlisted'].reset_index(drop=True)

counts = df_final['label'].value_counts()
print("Количество изображений по классам:\n", counts)

valid_labels = counts[counts >= 10].index
df_final = df_final[df_final['label'].isin(valid_labels)].reset_index(drop=True)
le = LabelEncoder()
df_final['label'] = le.fit_transform(df_final['color'])
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df_final['label']),
    y=df_final['label']
)

print(f"После фильтрации осталось классов: {len(df_final['label'].unique())}")
print(f"Всего изображений: {len(df_final)}")

Количество изображений по классам:
 label
1     14317
8      9474
21     9395
2      8483
18     7770
17     6095
4       911
7       777
22      667
0       600
14      559
16      362
3       329
6       217
12      196
15       87
19       26
11       26
10        9
5         9
13        1
9         1
Name: count, dtype: int64
После фильтрации осталось классов: 18
Всего изображений: 60291


In [4]:
train_df, test_df = train_test_split(df_final, test_size=0.2, stratify=df_final['label'], random_state=0)

class CarDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['img_path']
        label = self.df.iloc[idx]['label']
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_loader = DataLoader(
    CarDataset(train_df, train_transform),
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    CarDataset(test_df, test_transform),
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [5]:
class SimpleColorNet(nn.Module):
    def __init__(self, num_classes):
        super(SimpleColorNet, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = df_final['label'].nunique()
model = SimpleColorNet(num_classes).to(device)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

In [6]:
def train_model(model, optimizer, criterion, train_loader, test_loader, epochs=10, name="Model"):
    best_f1 = 0
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"{name} Epoch {epoch+1}")
        
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
        model.eval()
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        
        if f1 > best_f1:
            best_f1 = f1
        
        print(f"{name} Epoch {epoch+1} | Loss: {running_loss/len(train_loader):.4f} | F1: {f1:.4f}")
    
    return best_f1

In [7]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

best_f1_simple = train_model(
    model, optimizer, criterion,
    train_loader, test_loader,
    name="SimpleCNN"
)

SimpleCNN Epoch 1: 100%|██████████| 754/754 [09:13<00:00,  1.36it/s, loss=2.1594]


SimpleCNN Epoch 1 | Loss: 2.1291 | F1: 0.1885


SimpleCNN Epoch 2: 100%|██████████| 754/754 [04:21<00:00,  2.89it/s, loss=1.2644]


SimpleCNN Epoch 2 | Loss: 1.8634 | F1: 0.3058


SimpleCNN Epoch 3: 100%|██████████| 754/754 [04:19<00:00,  2.90it/s, loss=1.6083]


...

SimpleCNN Epoch 8 | Loss: 1.6247 | F1: 0.3222


SimpleCNN Epoch 9: 100%|██████████| 754/754 [03:41<00:00,  3.40it/s, loss=1.1208]


SimpleCNN Epoch 9 | Loss: 1.5895 | F1: 0.3056


SimpleCNN Epoch 10: 100%|██████████| 754/754 [03:41<00:00,  3.40it/s, loss=1.6614]


SimpleCNN Epoch 10 | Loss: 1.5762 | F1: 0.3612


In [8]:
model_resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

num_ftrs = model_resnet.fc.in_features
model_resnet.fc = nn.Linear(num_ftrs, num_classes)
model_resnet = model_resnet.to(device)

for param in model_resnet.parameters():
    param.requires_grad = False
for param in model_resnet.layer4.parameters():
    param.requires_grad = True
for param in model_resnet.fc.parameters():
    param.requires_grad = True

optimizer_res = optim.Adam(model_resnet.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

best_f1_resnet = train_model(
    model_resnet, optimizer_res, criterion,
    train_loader, test_loader,
    name="ResNet"
)

ResNet Epoch 1: 100%|██████████| 754/754 [02:52<00:00,  4.36it/s, loss=1.0160]


ResNet Epoch 1 | Loss: 1.3883 | F1: 0.4791


ResNet Epoch 2: 100%|██████████| 754/754 [02:52<00:00,  4.37it/s, loss=0.6713]


ResNet Epoch 2 | Loss: 0.8701 | F1: 0.5020


ResNet Epoch 3: 100%|██████████| 754/754 [02:51<00:00,  4.39it/s, loss=0.5030]


...

ResNet Epoch 8 | Loss: 0.2885 | F1: 0.5853


ResNet Epoch 9: 100%|██████████| 754/754 [02:53<00:00,  4.35it/s, loss=0.8333]


ResNet Epoch 9 | Loss: 0.2503 | F1: 0.6028


ResNet Epoch 10: 100%|██████████| 754/754 [03:10<00:00,  3.95it/s, loss=0.0936]


ResNet Epoch 10 | Loss: 0.2304 | F1: 0.6278


In [9]:
model_mobilenet = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)

num_ftrs = model_mobilenet.classifier[3].in_features
model_mobilenet.classifier[3] = nn.Linear(num_ftrs, num_classes)
model_mobilenet = model_mobilenet.to(device)

for param in model_mobilenet.parameters():
    param.requires_grad = False
for param in model_mobilenet.classifier.parameters():
    param.requires_grad = True

optimizer_mobile = optim.Adam(model_mobilenet.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

best_f1_mobile = train_model(
    model_mobilenet, optimizer_mobile, criterion,
    train_loader, test_loader,
    name="MobileNet"
)

MobileNet Epoch 1: 100%|██████████| 754/754 [03:01<00:00,  4.16it/s, loss=1.7218]


MobileNet Epoch 1 | Loss: 2.0484 | F1: 0.3476


MobileNet Epoch 2: 100%|██████████| 754/754 [03:10<00:00,  3.96it/s, loss=2.3234]


MobileNet Epoch 2 | Loss: 1.4936 | F1: 0.3730


MobileNet Epoch 3: 100%|██████████| 754/754 [02:58<00:00,  4.22it/s, loss=0.8539]


...

MobileNet Epoch 8 | Loss: 1.0677 | F1: 0.4214


MobileNet Epoch 9: 100%|██████████| 754/754 [02:56<00:00,  4.26it/s, loss=1.1129]


MobileNet Epoch 9 | Loss: 1.0220 | F1: 0.4232


MobileNet Epoch 10: 100%|██████████| 754/754 [02:56<00:00,  4.28it/s, loss=1.7326]


MobileNet Epoch 10 | Loss: 1.0180 | F1: 0.4283


In [10]:
print("\nЛучшие метрики f1")
print(f"SimpleCNN: {best_f1_simple:.4f}")
print(f"ResNet: {best_f1_resnet:.4f}")
print(f"MobileNet: {best_f1_mobile:.4f}")


Лучшие метрики f1
SimpleCNN: 0.3612
ResNet: 0.6278
MobileNet: 0.4283
